In [ ]:
##install and import necessary modules
##this code was originally designed and run in google colab
##use outside of colab may require modification
##if using colab, you may need to restart your runtime after installing modules,
##depending on enviornment at time of code running.

!pip install scikit-learn==1.5.2
!pip install tensorflow==2.12.1
!pip install xgboost==2.0.2
!pip install shap
!pip install statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import seaborn as sn
import sys
import sklearn
import statsmodels.api as sm
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils import resample
from scipy.stats import mannwhitneyu
from IPython import display
from pandas.api.types import is_numeric_dtype
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, recall_score, confusion_matrix, brier_score_loss, f1_score

sn.set(style='whitegrid')
pd.set_option('display.max_columns', None)

print("Python version:", sys.version)
print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgb.__version__)
print("shap version:", shap.__version__)

In [ ]:
# #import your dataset
# ##mount google drive if using in colab. Replace <MOUNT_POINT> with the directory where you want to mount the drive (e.g., /content/drive).
drive.mount('<MOUNT_POINT>')

# # Replace <YOUR_FILE_PATH> with the actual path inside your Google Drive (e.g., My Drive/FileNameHere).
file_path = '<MOUNT_POINT>/<YOUR_FILE_PATH>.csv'

In [ ]:
##specify columns to load from your dataset.  We will only load the columns necessary for the score and the variables necessary for the ISS/TRISS comparisons

columns_to_load = ['AGEYEARS', 'TOTALGCS', 'SBP'
                  , 'TEMPERATURE'
                  , 'PULSERATE', 'TRISS', 'TRISS_Death', 'MORTALITY', 'TRAUMATYPE'
                  , 'WEIGHT'
                  , 'ISS_05', 'NumberOfInjuriesOld', "FacilityKey",
                   'IntracranialVascularInjury','BrainStemInjury','EDH','SAH','SDH','SkullFx','DAI','NeckVascularInjury','ThoracicVascularInjury','AeroDigestiveInjury',
                   'CardiacInjury','LungInjury','AbdominalVascular','RibFx','KidneyInjury','StomachInjury','SpleenInjury','UroGenInternalInjury','SCI','SpineFx',
                   'UEAmputation','UEVascularInjury','UELongBoneFx','LEVascularInjury','PelvicFx','LEAmputation','PancreasInjury','LELongBoneFx','LiverInjury',
                   'ColorectalInjury','SmallBowelInjury','IPH', 'RESPIRATORYRATE'
                   ,'SEX', 'PRIMARYMETHODPAYMENT', 'RACE', 'ETHNICITY'
                   ]

columns_to_load_test = ['AGEYEARS', 'TOTALGCS', 'SBP'
                  , 'TEMPERATURE'
                  , 'PULSERATE', 'MORTALITY', 'TRAUMATYPE'
                  , 'WEIGHT'
                  , 'ISS_05', 'NumberOfInjuries',
                   'IntracranialVascularInjury','BrainStemInjury','EDH','SAH','SDH','SkullFx','DAI','NeckVascularInjury','ThoracicVascularInjury','AeroDigestiveInjury',
                   'CardiacInjury','LungInjury','AbdominalVascular','RibFx','KidneyInjury','StomachInjury','SpleenInjury','UroGenInternalInjury','SCI','SpineFx',
                   'UEAmputation','UEVascularInjury','UELongBoneFx','LEVascularInjury','PelvicFx','LEAmputation','PancreasInjury','LELongBoneFx','LiverInjury',
                   'ColorectalInjury','SmallBowelInjury','IPH', "RevisedTraumaScore", 'RespiratoryRate'
                   ,'SEX', 'PrimaryPayor', 'RACE', 'ETHNICITY'
                   ]

In [ ]:
# Import data and specify missing values (both training and test sets)
data = pd.read_csv(file_path, na_values=['NA', 'N/A', 'NULL', ' ', '', '-99', '-98', '-99.0', '-99.00', '-98.0', '-98.00', 'NaN'], usecols=columns_to_load)
data_test = pd.read_csv(file_path_test, na_values=["*BL",'NA', 'N/A', 'NULL', ' ', '', '-99', '-98', '-99.0', '-99.00', '-98.0', '-98.00', 'NaN'], usecols=columns_to_load_test)



##ensure SBP and HR are of datatype numeric
for df in (data, data_test):
    for c in ['SBP', 'PULSERATE']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')


# Filter out rows where 'TRAUMATYPE' is not "Blunt" or "Penetrating"
try:
  exclude_values = ['26', 'Other/unspecified', 'Burn']
  data = data[~data['TRAUMATYPE'].isin(exclude_values)]
except:
  pass

##compute TRISS
b0_blunt=-1.2470
b0_pen=-0.6029
b1_blunt=0.9544
b1_pen=	1.1430
b2_blunt=-0.0768
b2_pen=-0.1516
b3_blunt=-1.9052
b3_pen=-2.6676

## Age indicator (A)
A = np.where(data_test['AGEYEARS'].isna(), np.nan,
             (data_test['AGEYEARS'] > 54).astype(int))

## choose coefficients based on trauma type
# --- Trauma type parsing: missing trauma -> NaN; unknown strings -> NaN ---
tt = data_test['TRAUMATYPE'].astype('string').str.strip().str.lower()

is_pen = tt.eq('penetrating') | tt.eq('pen')
is_blunt = tt.eq('blunt') | tt.eq('bl')

# anything not recognized becomes missing
valid = is_pen | is_blunt

b0 = np.where(~valid, np.nan, np.where(is_pen, b0_pen, b0_blunt))
b1 = np.where(~valid, np.nan, np.where(is_pen, b1_pen, b1_blunt))
b2 = np.where(~valid, np.nan, np.where(is_pen, b2_pen, b2_blunt))
b3 = np.where(~valid, np.nan, np.where(is_pen, b3_pen, b3_blunt))

## linear predictor
b = (
    b0
    + b1 * data_test['RevisedTraumaScore']
    + b2 * data_test['ISS_05']
    + b3 * A
)

## probability of survival
data_test['TRISS'] = (1 / (1 + np.exp(-b))).clip(0, 1)
data_test['TRISS_Death'] = (1 - data_test['TRISS']).clip(0, 1)

##explicitly list variables that need to be present for inclusion and drop cases without these
##we cannot compare our score to ISS/TRISS without those metrics, and we need our target outcome mortality
required_vars = ['ISS_05', 'TRISS_Death', 'MORTALITY', 'SBP', 'TOTALGCS']
data = data.dropna(subset=required_vars)
data_test = data_test.dropna(subset=required_vars)

##dropRTS from test set
data_test = data_test.drop(columns=['RevisedTraumaScore'])

# Create ShockIndex with the required logic
data['ShockIndex'] = np.where(
    data['SBP'] == 0, 2.0,  # Case where SBP is 0 → set ShockIndex to 2.0
    data['PULSERATE'] / data['SBP']  # Normal calculation
)
data_test['ShockIndex'] = np.where(
    data_test['SBP'] == 0, 2.0,  # Case where SBP is 0 → set ShockIndex to 2.0
    data_test['PULSERATE'] / data_test['SBP']  # Normal calculation
)

# Set ShockIndex to NaN if PULSERATE or SBP is missing
data.loc[data['PULSERATE'].isna() | data['SBP'].isna(), 'ShockIndex'] = np.nan
data_test.loc[data_test['PULSERATE'].isna() | data_test['SBP'].isna(), 'ShockIndex'] = np.nan

##reset indices of the df
data.reset_index(drop=True, inplace=True)
data_test.reset_index(drop=True, inplace=True)

In [ ]:
# Create binary variables for ISS >=30
iss30_train = (data['ISS_05'] >= 30).astype(int)
iss30_test = (data_test['ISS_05'] >= 30).astype(int)

# Reuse your function (just change positive_value to 1)
iss30_smd = smd_binary(iss30_train, iss30_test, positive_value='1')

print(f"ISS ≥30 SMD = {iss30_smd:.3f}")

In [ ]:
##Rename variable from dataset upload
data = data.rename(columns={'NumberOfInjuriesOld': 'NumberOfInjuries'})

In [ ]:
##Rename variable from dataset upload
data_test = data_test.rename(columns={'PrimaryPayor': 'PRIMARYMETHODPAYMENT'})

In [ ]:
##Drop variables needed for RTS calculations that aren't used in the model
data=data.drop(columns=["RESPIRATORYRATE"])

In [ ]:
##Drop variables needed for RTS calculations that aren't used in the model (test set)
data_test=data_test.drop(columns=["RespiratoryRate"])

In [ ]:
##verify data appears as intended (training)
data.head()

In [ ]:
##verify data_test appears as intended (test)
data_test.head()

In [ ]:
##check for missing values (training)
data.isnull().sum(axis=0)

In [ ]:
##check for missing values (test)
data_test.isnull().sum(axis=0)

In [ ]:
##create a datafram of all complications/vars to remove later.  We can remove all of these from the X data set and pick one to be
#our Y dataset

complications_df=pd.DataFrame()
complications_list= [
                    'MORTALITY', 'TRISS', 'SEX', 'PRIMARYMETHODPAYMENT', 'RACE', 'ETHNICITY', 'ISS_05'
                    ]
for c in complications_list:
    complications_df[c] = data_test[c]
complications_df

In [ ]:
##this is where we choose our outcome variable, mortality, and give it its own dataframe

Y_data = pd.DataFrame()
Y_data['MORTALITY'] = data['MORTALITY']
Y_data

In [ ]:
##now do the same thing for test set
Y_data_test = pd.DataFrame()
Y_data_test['MORTALITY'] = data_test['MORTALITY']
Y_data_test

In [ ]:
##clean Y_data by replacing "Yes" and "No" vcalues with 0's and 1's

Y_data['MORTALITY'] = Y_data['MORTALITY'].replace({'Yes': 1, 'No': 0})
Y_data

In [ ]:
##clean Y_data_test by replacing "Yes" and "No" vcalues with 0's and 1's

Y_data_test['MORTALITY'] = Y_data_test['MORTALITY'].replace({'Yes': 1, 'No': 0})
Y_data_test

In [ ]:
##now drop the outcome from our feature space as well as TRISS, since were using 1-TRISS (aka TRISS_Death) and this varibale is now useless,
##as well as vars used only for stratifying into groups but not for model
X_data = data.drop(columns=['MORTALITY', 'TRISS', 'SEX', 'PRIMARYMETHODPAYMENT', 'RACE', 'ETHNICITY'])
X_data.shape

In [ ]:
##same as above but for test
X_data_test = data_test.drop(columns=['MORTALITY', 'TRISS', 'SEX', 'PRIMARYMETHODPAYMENT', 'RACE', 'ETHNICITY'])
X_data_test.shape

In [ ]:
##ensure no missing outcome data
Missing_Y = Y_data.isnull().sum(axis=0)
Missing_Y

In [ ]:
##ensure no missing outcome data (test)
Missing_Y_test = Y_data_test.isnull().sum(axis=0)
Missing_Y_test

In [ ]:
##If we have no missing values here, our data is clean
Y_clean=Y_data
del Y_data

In [ ]:
##If we have no missing values here, our data is clean (test)
Y_clean_test=Y_data_test
del Y_data_test

In [ ]:
##if above check passes, outcome data is now clean
Missing_Y_clean = Y_clean.isnull().sum(axis=0)
Missing_Y_clean

In [ ]:
##if above check passes, outcome data is now clean (test)
Missing_Y_clean_test = Y_clean_test.isnull().sum(axis=0)
Missing_Y_clean_test

In [ ]:
##check which variables in the input space have missing variables

Missing = X_data.isnull().sum(axis=0)
Missing[Missing>0]

In [ ]:
##check which variables in the input space have missing variables (test)

Missing_test = X_data_test.isnull().sum(axis=0)
Missing_test[Missing_test>0]

In [ ]:
##order variables with missing data by percentage

data_missing = (X_data.isnull().sum(axis=0)/X_data.shape[0]) * 100
data_missing

In [ ]:
##order variables with missing data by percentage (test)

data_missing_test = (X_data_test.isnull().sum(axis=0)/X_data_test.shape[0]) * 100
data_missing_test

In [ ]:
##display variables withOUT mising data

data_missing[data_missing == 0].index

In [ ]:
##display variables withOUT mising data (test)

data_missing_test[data_missing_test == 0].index

In [ ]:
#remove the good columns (no missing values) from data_missing

data_missing = data_missing.drop(data_missing[data_missing == 0].index)
data_missing

In [ ]:
#remove the good columns (no missing values) from data_missing_test

data_missing_test = data_missing_test.drop(data_missing_test[data_missing_test == 0].index)
data_missing_test

In [ ]:
#sort this in ascending order
data_missing = data_missing.sort_values(ascending=False)
data_missing

In [ ]:
#sort this in ascending order (test)
data_missing_test = data_missing_test.sort_values(ascending=False)
data_missing_test

In [ ]:
##prepare to drop variables with >50% missing values

dropCutoff=50
bad_column_names = data_missing[data_missing >=dropCutoff].index
bad_column_names

In [ ]:
##check which variables are above this >50% missingness cutoff
bad_column_names_test = data_missing_test[data_missing_test >=dropCutoff].index
bad_column_names_test

In [ ]:
##actually drop bad variables
X_data_new=X_data.drop(columns=bad_column_names, axis=1)

##check for which variables still have missing data (<50% missing values)
Missing = X_data_new.isnull().sum(axis=0)
Missing[Missing>0]

In [ ]:
##actually drop bad variables (test)
X_data_new_test=X_data_test.drop(columns=bad_column_names, axis=1)

##check for which variables still have missing data (<50% missing values)
Missing_test = X_data_new_test.isnull().sum(axis=0)
Missing_test[Missing_test>0]

In [ ]:
#display columns with less than 50% missing that need to be cleaned

to_be_cleaned_column_names = data_missing[data_missing <50].index
to_be_cleaned_column_names

In [ ]:
#display columns with less than 50% missing that need to be cleaned (test)

to_be_cleaned_column_names_test = data_missing_test[data_missing_test <50].index
to_be_cleaned_column_names_test

In [ ]:
# Rename the 'TRAUMATYPE' column to 'Penetrating' and map the values to 0 and 1
X_data_new['Penetrating'] = X_data_new['TRAUMATYPE'].map({'Penetrating': 1, 'Blunt': 0})

# Drop the old 'TRAUMATYPE' column
X_data_new.drop(columns=['TRAUMATYPE'], inplace=True)

print(X_data_new.head())

In [ ]:
#same thing for test set
# Rename the 'TRAUMATYPE' column to 'Penetrating' and map the values to 0 and 1
X_data_new_test['Penetrating'] = X_data_new_test['TRAUMATYPE'].map({'Penetrating': 1, 'Blunt': 0})

# Drop the old 'TRAUMATYPE' column
X_data_new_test.drop(columns=['TRAUMATYPE'], inplace=True)

print(X_data_new_test.head())

In [ ]:
# Display the entire DataFrame without truncation
pd.set_option('display.max_columns', None)

# Get column names and data types
columns_info = []
for column_name, dtype in zip(X_data_new.columns, X_data_new.dtypes):
    columns_info.append(f"{column_name}: {dtype}")

formatted_columns_info = "\n".join(columns_info)

# Print column names and data types
print("Column Names and Data Types:")
print(formatted_columns_info)

In [ ]:
# Display the entire DataFrame without truncation (now for test)
pd.set_option('display.max_columns', None)

# Get column names and data types
columns_info_test = []
for column_name, dtype in zip(X_data_new_test.columns, X_data_new_test.dtypes):
    columns_info_test.append(f"{column_name}: {dtype}")

formatted_columns_info_test = "\n".join(columns_info)

# Print column names and data types
print("Column Names and Data Types:")
print(formatted_columns_info_test)

In [ ]:
##convert No's and Yes's to 0's and 1's to minimize the amount of double variables (want to avoid Yes/Nos being converted to 1-hot variables)

try:
    X_data_new= X_data_new.replace({True: 1, 'Yes': 1, "Female": 1, False: 0, 'No': 0, "Male": 0})
except:
    pass

X_data_new.head()

In [ ]:
##convert No's and Yes's to 0's and 1's to minimize the amount of double variables (want to avoid Yes/Nos being converted to 1-hot variables) (test set)

try:
    X_data_new_test= X_data_new_test.replace({True: 1, 'Yes': 1, "Female": 1, False: 0, 'No': 0, "Male": 0})
except:
    pass

X_data_new_test.head()

In [ ]:
#now prepare to do train/test splitting at the facility level

# total number of unique facilities
num_facilities = data["FacilityKey"].nunique()
print(f"Total number of facilities: {num_facilities}")

# build table of facility case counts
facility_counts = (
    data["FacilityKey"]
    .value_counts()
    .reset_index(name="CaseCount")       # force the count column name
    .rename(columns={"index": "FacilityKey"})
    .sort_values("CaseCount", ascending=False)   # sort by size
    .reset_index(drop=True)              # clean index after sorting
)

# print all rows without truncation
with pd.option_context("display.max_rows", None):
    print(facility_counts)

# delete intermediate df
del facility_counts

In [ ]:
##verify correct data shape
X_data_new.shape

In [ ]:
##preppare for hospital level splitting with various helper functions

def _extract_y_vector(y, y_col):
    if isinstance(y, pd.Series):
        return y
    if isinstance(y, pd.DataFrame):
        return y[y_col] if y_col is not None else (
            y.iloc[:, 0] if y.shape[1] == 1 else
            (_ for _ in ()).throw(ValueError("Y_clean has multiple columns; provide y_col."))
        )
    raise TypeError("Y_clean must be a pandas Series or DataFrame.")

def make_hospital_splits_Xy_train_cal(
    X, Y,
    facility_col="FacilityKey",
    y_col=None,
    train_prop=0.80, cal_prop=0.20,
    mega_quantile=0.95,
    n_bins=10,
    handle_missing_facility="drop",  # 'drop' or 'assign_train'
    seed=42
):
    assert abs(train_prop + cal_prop - 1.0) < 1e-9, "Proportions must sum to 1."

    # ---- Align X and Y on common index ----
    common_idx = X.index.intersection(Y.index)
    if (len(common_idx) != len(X)) or (len(common_idx) != len(Y)):
        print(f"[info] Aligning X and Y to common index: {len(common_idx)} rows kept (X={len(X)}, Y={len(Y)}).")
    X = X.loc[common_idx].copy()
    Y = Y.loc[common_idx].copy()

    # ---- Facility column (from X if present else Y) ----
    if facility_col in X.columns:
        fac = X[facility_col].copy()
    elif facility_col in Y.columns:
        fac = Y[facility_col].copy()
    else:
        raise KeyError(f"'{facility_col}' not found in X_data_new or Y_clean.")
    fac.name = facility_col

    # ---- Handle missing FacilityKey rows ----
    n_missing = fac.isna().sum()
    if n_missing > 0:
        if handle_missing_facility == "drop":
            keep = fac.notna()
            print(f"[warn] Dropping {n_missing} rows with missing {facility_col}.")
            X, Y, fac = X.loc[keep], Y.loc[keep], fac.loc[keep]
        elif handle_missing_facility == "assign_train":
            fac = fac.fillna("__UNKNOWN__")
        else:
            raise ValueError("handle_missing_facility must be 'drop' or 'assign_train'.")

    # ---- Outcome vector (optional, for summaries) ----
    y_vec = None
    try:
        y_vec = _extract_y_vector(Y, y_col).loc[X.index]
    except Exception:
        pass

    # ---- Hospital-level table ----
    hosp = fac.value_counts(dropna=False).rename_axis(facility_col).rename("n").to_frame()
    if y_vec is not None:
        ev = y_vec.groupby(fac).sum()
        hosp["events"] = ev.reindex(hosp.index).fillna(0).astype(int)
    else:
        hosp["events"] = np.nan

    # ---- Identify & balance mega hospitals ----
    thr = hosp["n"].quantile(mega_quantile)
    mega_idx = hosp[hosp["n"] >= thr].sort_values("n", ascending=False).index
    rest_idx = hosp.index.difference(mega_idx)

    splits = {h: None for h in hosp.index}
    load = {"train": 0, "cal": 0}
    order = ["train", "cal"]

    for h in mega_idx:
        target = min(load, key=load.get)
        ties = [k for k, v in load.items() if v == load[target]]
        if len(ties) > 1:
            for name in order:
                if name in ties:
                    target = name
                    break
        splits[h] = target
        load[target] += int(hosp.loc[h, "n"])

    # ---- Stratify remaining by log-volume and sample ----
    rng = np.random.default_rng(seed)
    if len(rest_idx) > 0:
        rest = hosp.loc[rest_idx].copy()
        rest["log_n"] = np.log1p(rest["n"])
        rest["vol_bin"] = pd.qcut(rest["log_n"], q=n_bins, duplicates="drop")

        for _, sub in rest.groupby("vol_bin", observed=True):
            ids = sub.index.to_numpy()
            rng.shuffle(ids)
            k = len(ids)

            n_train = int(round(train_prop * k))
            n_cal   = k - n_train  # remainder to cal

            # small-bin safeguard: try to ensure both non-empty when possible
            if k >= 2:
                if n_train == 0:
                    n_train = 1
                    n_cal = k - n_train
                if n_cal == 0:
                    n_cal = 1
                    n_train = k - n_cal

            assign = {
                "train": ids[:n_train],
                "cal":   ids[n_train:]
            }
            for name, arr in assign.items():
                for h in arr:
                    splits[h] = name

    # ---- Fallback: assign any leftover hospitals (None) to smallest-load split ----
    leftover = [h for h, s in splits.items() if s is None]
    if leftover:
        print(f"[warn] {len(leftover)} hospital(s) were unassigned after stratification; assigning by load balance.")
        for h in leftover:
            target = min(load, key=load.get)
            splits[h] = target
            load[target] += int(hosp.loc[h, "n"])

    # ---- Build mapping and assign rows ----
    split_map = pd.DataFrame({facility_col: list(splits.keys()), "split": list(splits.values())})
    assert split_map["split"].isna().sum() == 0, "Some hospitals were not assigned (unexpected)."

    split_series = fac.map(split_map.set_index(facility_col)["split"])

    X_train = X[split_series.eq("train")].copy()
    X_cal   = X[split_series.eq("cal")].copy()

    Y_train = Y.loc[X_train.index].copy()
    Y_cal   = Y.loc[X_cal.index].copy()

    # ---- Summaries ----
    def _summary(ix, name):
        nrows = len(ix)
        uh = fac.loc[ix].nunique(dropna=False)
        if y_vec is not None:
            ev = int(y_vec.loc[ix].sum())
            print(f"{name:>5} | rows={nrows:,} | hospitals={uh:,} | events={ev:,}")
        else:
            print(f"{name:>5} | rows={nrows:,} | hospitals={uh:,}")

    print("\n=== Split summary (rows / hospitals / events) ===")
    _summary(X_train.index, "train")
    _summary(X_cal.index,   "cal")

    return X_train, X_cal, Y_train, Y_cal, split_map, hosp.sort_values("n", ascending=False)

In [ ]:
## split into train + calibration sets (facility-level)

X_train_cal, X_val_cal, Y_train_cal, Y_val_cal, split_map, hosp_table = make_hospital_splits_Xy_train_cal(
    X_data_new, Y_clean,
    facility_col="FacilityKey",
    y_col="MORTALITY",      # omit or set None if Y_clean is a Series already
    train_prop=0.80, cal_prop=0.20,
    mega_quantile=0.95, n_bins=10, seed=0
)

In [ ]:
##Lets delete some intermediate variablen assignments from memory to free up ram

X_test = X_data_new_test
Y_test = Y_clean_test

del X_data_new_test
del Y_clean_test
del data_test
del data

import gc
gc.collect()

In [ ]:
# Drop FacilityKey from X predictors
for df in (X_train_cal, X_val_cal):
    df.drop(columns=["FacilityKey"], inplace=True, errors="ignore")


In [ ]:
# before imputation, lets figure out which vars have missing data that need to be imputed
to_be_cleaned_column_names = [c for c in to_be_cleaned_column_names if c != 'FacilityKey']

In [ ]:
# before imputation (test set)
to_be_cleaned_column_names_test = [c for c in to_be_cleaned_column_names_test if c != 'FacilityKey']

In [ ]:
##now design helper function to do median/mode imputation


def impute_from_train(X_train_cal, X_val_cal, to_be_cleaned_column_names):
    for c in to_be_cleaned_column_names:
        s_train = X_train_cal[c]

        if is_numeric_dtype(s_train):
            median_value = s_train.median(skipna=True)
            if pd.isna(median_value):
                raise ValueError(f"All values missing in TRAIN for numeric column '{c}'; cannot compute median.")
            for df in (X_train_cal, X_val_cal):
                df.loc[:, c] = df[c].fillna(median_value)
        else:
            mode_series = s_train.mode(dropna=True)
            if mode_series.empty:
                raise ValueError(f"All values missing in TRAIN for categorical column '{c}'; cannot compute mode.")
            mode_value = mode_series.iloc[0]
            for df in (X_train_cal, X_val_cal):
                df.loc[:, c] = df[c].astype(object).fillna(mode_value)

# actually used function
impute_from_train(X_train_cal, X_val_cal, to_be_cleaned_column_names)

In [ ]:
##same thing as above for test set

def impute_from_train_test(X_train_cal, X_test, to_be_cleaned_column_names_test):
    for c in to_be_cleaned_column_names_test:
        s_train = X_train_cal[c]

        if is_numeric_dtype(s_train):
            median_value = s_train.median(skipna=True)
            if pd.isna(median_value):
                raise ValueError(f"All values missing in TRAIN for numeric column '{c}'; cannot compute median.")
            for df in (X_train_cal, X_test):
                df.loc[:, c] = df[c].fillna(median_value)
        else:
            mode_series = s_train.mode(dropna=True)
            if mode_series.empty:
                raise ValueError(f"All values missing in TRAIN for categorical column '{c}'; cannot compute mode.")
            mode_value = mode_series.iloc[0]
            for df in (X_train_cal, X_test):
                df.loc[:, c] = df[c].astype(object).fillna(mode_value)

# usage
impute_from_train_test(X_train_cal, X_test, to_be_cleaned_column_names_test)

In [ ]:
#recompute shock index after imputation to ensure no mismatched HR/SBP vars with SI var
def recompute_shock_index(df, pulse_col='PULSERATE', sbp_col='SBP', out_col='ShockIndex'):
    # Ensure numeric (in case dtype got set to object somewhere)
    df.loc[:, pulse_col] = pd.to_numeric(df[pulse_col], errors='coerce')
    df.loc[:, sbp_col]   = pd.to_numeric(df[sbp_col],   errors='coerce')

    # ShockIndex = 2.0 if SBP == 0, else PULSERATE / SBP
    df.loc[:, out_col] = np.where(df[sbp_col] == 0, 2.0, df[pulse_col] / df[sbp_col])

# After calling your impute_from_train(...):
recompute_shock_index(X_train_cal)
recompute_shock_index(X_val_cal)
recompute_shock_index(X_test)


In [ ]:
##now for one-hot encoding

# Identify categorical columns from X_train only
categorical_column = [c for c in X_train_cal.columns if X_train_cal[c].dtype == np.dtype('O')]

# Apply pd.get_dummies to all data sets
X_train_cal = pd.get_dummies(X_train_cal, columns=categorical_column, sparse=False)
X_test = pd.get_dummies(X_test, columns=categorical_column, sparse=False)
X_val_cal = pd.get_dummies(X_val_cal, columns=categorical_column, sparse=False)

categorical_column

In [ ]:
#verify data appears as intended
X_train_cal.head()

In [ ]:
##verify no missing data in any split dataset
print(X_train_cal.isnull().sum().sum())
print(X_test.isnull().sum().sum())
print(X_val_cal.isnull().sum().sum())

In [ ]:
##final list of training columns
X_train_cal.columns

In [ ]:
##final list of training columns (test)
X_test.columns

In [ ]:
#verify data is intended size
X_test.shape

In [ ]:
##now with data cleaned, take comparison vars and move them to their own dataframe prior to dropping
new_to_drop = ['TRISS_Death', 'ISS_05']

X_ISS=pd.DataFrame()
X_ISS['ISS']=X_test['ISS_05']

X_TRISS=pd.DataFrame()
X_TRISS['TRISS']=X_test['TRISS_Death']

In [ ]:
##now drop those comparison vars from the data that will be fed to the model
X_train_cal.drop(columns=new_to_drop, inplace=True)
X_test.drop(columns=new_to_drop, inplace=True)
X_val_cal.drop(columns=new_to_drop, inplace=True)

In [ ]:
##forcing the test dataset to have exactly the same columns, in the same order, as the training/calibration dataset.
X_test = X_test.reindex(columns=X_train_cal.columns)

In [ ]:
##label all vars in the dataset as either float vars or biinary vars
float32_cols = [
    'AGEYEARS', 'SBP', 'PULSERATE', 'TEMPERATURE', 'WEIGHT',
    'TOTALGCS', 'ISS_05', 'NumberOfInjuries', 'TRISS_Death', 'ShockIndex'
]

binary_cols = [
    'IntracranialVascularInjury', 'BrainStemInjury', 'EDH', 'SAH', 'SDH', 'IPH',
    'SkullFx', 'DAI', 'NeckVascularInjury', 'ThoracicVascularInjury',
    'AeroDigestiveInjury', 'CardiacInjury', 'LungInjury', 'AbdominalVascular',
    'RibFx', 'KidneyInjury', 'StomachInjury', 'SpleenInjury',
    'UroGenInternalInjury', 'SCI', 'SpineFx', 'UEAmputation',
    'UEVascularInjury', 'UELongBoneFx', 'LEVascularInjury', 'PelvicFx',
    'LEAmputation', 'PancreasInjury', 'LELongBoneFx', 'LiverInjury',
    'ColorectalInjury', 'SmallBowelInjury', 'Penetrating', 'HemoPneumo'
]

##coerce continuous vars to be float 32 and binary to int8 to save memory
for df in (X_train_cal, X_test):
    for c in float32_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype('int8')

In [ ]:
##Next step is to normalize data

scaler=StandardScaler()
#get the parameters of the transform
scaler.fit(X_train_cal)

#normalize the features in the training set
X_train_s_cal = scaler.transform(X_train_cal)
#normalize the features in the test set
print("After train/test split, X_test shape:", X_test.shape)
X_test_s = scaler.transform(X_test)
print("After scaling, X_test_s shape:", X_test_s.shape)
#normalize the features in the val set
X_val_s_cal = scaler.transform(X_val_cal)

In [ ]:
##combine datasets now that youved used only X_train to develop the scaler to avoid using calibration data for scaling the training data
X_dev_full = pd.concat([X_train_cal, X_val_cal], axis=0)\

In [ ]:
##now, fit model with hyperparameters based on other Jupyternotebook optimization
model_best_gb = xgb.XGBClassifier(random_state=0, colsample_bytree=0.6, learning_rate=0.1, max_depth=7, n_estimators=200, subsample=1.0)
model_best_gb.fit(X_train_s_cal, Y_train_cal)

In [ ]:
# Get predicted probabilities for test set (evaluate model)
from sklearn.metrics import roc_curve, auc, precision_recall_curve, recall_score, confusion_matrix

y_prob_gbo_mtp = model_best_gb.predict_proba(X_test_s)[:, 1]

# Compute AUROC on test set
auroc_gbo = roc_auc_score(Y_test, y_prob_gbo_mtp)
print(f"AUROC on the test set: {auroc_gbo}")

In [ ]:
# Calibrate the model on the validation set
calibrated_model = CalibratedClassifierCV(estimator=model_best_gb, method='isotonic', cv='prefit')
calibrated_model.fit(X_val_s_cal, Y_val_cal)

In [ ]:
# Get predicted probabilities for test set (evaluate model)
y_prob_gbo_mtp = calibrated_model.predict_proba(X_test_s)[:, 1]

# Compute AUROC on test set (calibrated)
auroc_gbo = roc_auc_score(Y_test, y_prob_gbo_mtp)
print(f"AUROC on the test set: {auroc_gbo}")

In [ ]:
# Fit logistic regression: Outcome ~ ISS
lr_iss = LogisticRegression()
lr_iss.fit(X_ISS.values.reshape(-1,1), Y_test)

# Predict probabilities
iss_probs = lr_iss.predict_proba(X_ISS.values.reshape(-1,1))[:,1]

In [ ]:
##create a function we can re-use to compare two groups to one another
def evaluate_subgroup(mask, X_test_tensor, complications_df, calibrated_model, threshold=0.5):
    """
    Evaluate model performance for a specified subgroup (e.g., males or females).

    Parameters:
    - mask: boolean mask from complications_df
    - X_test_tensor: full test feature matrix (pandas DataFrame or array)
    - complications_df: full dataframe containing 'MORTALITY' and subgroup columns
    - calibrated_model: trained and calibrated model
    - threshold: classification threshold for computing F1, confusion matrix, etc.

    Returns:
    - Dictionary of performance metrics
    """
    # Subset the test features and labels
    X_sub = X_test_tensor[mask]
    y_true = complications_df.loc[mask, 'MORTALITY'].values
    y_prob = calibrated_model.predict_proba(X_sub)[:, 1]

    # Primary metrics
    auroc = roc_auc_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)

    # Threshold-dependent metrics
    y_pred = (y_prob >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred)

    # Optional confusion matrix-derived stats
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    npv = tn / (tn + fn) if (tn + fn) else 0

    return {
        "AUROC": auroc,
        "Brier Score": brier,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "Sensitivity (TPR)": sensitivity,
        "Specificity (TNR)": specificity,
        "Precision (PPV)": precision,
        "Negative Predictive Value (NPV)": npv
    }

In [ ]:
##allow for comparison between trauma severity scoring systems within a particular cohort, define helper fx

# === Define Paired Bootstrap Function ===
def paired_bootstrap_auc_test(
    y_true, predA, predB, n_boot=2000, alpha=0.025, random_state=None
):
    y_true = np.asarray(y_true)
    predA = np.asarray(predA)
    predB = np.asarray(predB)

    assert len(y_true) == len(predA) == len(predB), "Arrays must be the same length."
    n = len(y_true)

    aucA = roc_auc_score(y_true, predA)
    aucB = roc_auc_score(y_true, predB)
    baseline_diff = aucA - aucB

    rng = np.random.default_rng(random_state)
    aucAs = np.zeros(n_boot)
    aucBs = np.zeros(n_boot)
    diffs = np.zeros(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        try:
            aucAs[i] = roc_auc_score(y_true[idx], predA[idx])
            aucBs[i] = roc_auc_score(y_true[idx], predB[idx])
            diffs[i] = aucAs[i] - aucBs[i]
        except:
            aucAs[i] = aucBs[i] = diffs[i] = np.nan

    aucAs = aucAs[~np.isnan(diffs)]
    aucBs = aucBs[~np.isnan(diffs)]
    diffs = diffs[~np.isnan(diffs)]

    return {
        "aucA": aucA,
        "aucB": aucB,
        "aucA_ci_lower": np.percentile(aucAs, 100 * alpha),
        "aucA_ci_upper": np.percentile(aucAs, 100 * (1 - alpha)),
        "aucB_ci_lower": np.percentile(aucBs, 100 * alpha),
        "aucB_ci_upper": np.percentile(aucBs, 100 * (1 - alpha)),
        "baseline_diff": baseline_diff,
        "mean_diff": np.mean(diffs),
        "diff_ci_lower": np.percentile(diffs, 100 * alpha),
        "diff_ci_upper": np.percentile(diffs, 100 * (1 - alpha)),
        "p_value": min(1.0, 2 * min(np.mean(diffs < 0), np.mean(diffs > 0))),
        "coverage": (1 - 2 * alpha) * 100
    }


In [ ]:
##evaluate model within female and males separately

# Step 1: Prepare complications_test_df with mortality mapped to 0/1
complications_test_df = complications_df.loc[X_test.index].copy()
complications_test_df['MORTALITY'] = complications_test_df['MORTALITY'].map({'No': 0, 'Yes': 1})
complications_test_df = complications_test_df.dropna(subset=['MORTALITY'])
y_true_all = complications_test_df['MORTALITY'].astype(int).values

# Step 2: Generate model predictions (already trained + scaled)
# X_test_scaled = scaler.transform(X_test_tensor)
y_prob_model = y_prob_gbo_mtp

# Step 3: Add predictions to dataframe
complications_test_df['y_true'] = y_true_all
complications_test_df['y_prob_model'] = y_prob_model
complications_test_df['y_prob_iss'] = iss_probs.flatten()
complications_test_df['y_prob_triss'] = X_TRISS.values.flatten()

# Step 4: Evaluation function for any group
def evaluate_all_models(df, group_name):
    group_df = df[df['SEX'] == group_name]
    y_true = group_df['y_true']

    return {
        'Group': group_name,
        'AUROC_ML': roc_auc_score(y_true, group_df['y_prob_model']),
        'AUROC_ISS': roc_auc_score(y_true, group_df['y_prob_iss']),
        'AUROC_TRISS': roc_auc_score(y_true, group_df['y_prob_triss']),
        'Brier_ML': brier_score_loss(y_true, group_df['y_prob_model']),
        'Brier_ISS': brier_score_loss(y_true, group_df['y_prob_iss']),
        'Brier_TRISS': brier_score_loss(y_true, group_df['y_prob_triss']),
        'N': len(group_df),
        'Positives': int((y_true == 1).sum()),
        'Negatives': int((y_true == 0).sum())
    }

# Step 5: Run evaluation for Male and Female
results_male = evaluate_all_models(complications_test_df, 'Male')
results_female = evaluate_all_models(complications_test_df, 'Female')

# Step 6: Display results
for result in [results_male, results_female]:
    print(f"=== {result['Group']} ===")
    print(f"AUROC (ML):    {result['AUROC_ML']:.3f}")
    print(f"AUROC (ISS):   {result['AUROC_ISS']:.3f}")
    print(f"AUROC (TRISS): {result['AUROC_TRISS']:.3f}")
    print(f"Brier (ML):    {result['Brier_ML']:.3f}")
    print(f"Brier (ISS):   {result['Brier_ISS']:.3f}")
    print(f"Brier (TRISS): {result['Brier_TRISS']:.3f}")
    print(f"N:             {result['N']}")
    print(f"Positives:     {result['Positives']}")
    print(f"Negatives:     {result['Negatives']}\n")

    # Calculate AUROC delta between Male and Female for each method
delta_auroc_model = results_male['AUROC_ML'] - results_female['AUROC_ML']
delta_auroc_iss = results_male['AUROC_ISS'] - results_female['AUROC_ISS']
delta_auroc_triss = results_male['AUROC_TRISS'] - results_female['AUROC_TRISS']

# Print deltas
print("=== AUROC Deltas (Male - Female) ===")
print(f"Model: {delta_auroc_model:.4f}")
print(f"ISS:   {delta_auroc_iss:.4f}")
print(f"TRISS: {delta_auroc_triss:.4f}")


In [ ]:
##create male and female only DFs
female_df= complications_test_df[complications_test_df['SEX'] == 'Female']
male_df= complications_test_df[complications_test_df['SEX'] == 'Male']
female_df.shape

In [ ]:
##evaluate model within female and males separately
predicted_prob_iss_female=female_df['y_prob_iss']
predicted_prob_triss_female=female_df['y_prob_triss']
predicted_prob_gbo_female=female_df['y_prob_model']
true_label_female=female_df['y_true']

# Calculate the FPR, TPR, and thresholds
fpr_iss_female, tpr_iss_female, thresholds_iss_female = roc_curve(true_label_female, predicted_prob_iss_female)

##now TRISS
fpr_triss_female, tpr_triss_female, thresholds_triss_female = roc_curve(true_label_female, predicted_prob_triss_female)

##and MLISS
fpr_gbo_female, tpr_gbo_female, thresholds_gbo_female = roc_curve(true_label_female, predicted_prob_gbo_female)

# Calculate the area under the ROC curve (AUROC)
roc_auc_iss_female = auc(fpr_iss_female, tpr_iss_female)

##now TRISS
roc_auc_triss_female = auc(fpr_triss_female, tpr_triss_female)

##and MLISS
roc_auc_gbo_female = auc(fpr_gbo_female, tpr_gbo_female)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo_female, tpr_gbo_female, color='b', lw=2, label=f'ROC curve (area = {roc_auc_gbo_female:.3f})')
plt.plot(fpr_triss_female, tpr_triss_female, color='green', lw=2, label=f'ROC AUC TRISS = {roc_auc_triss_female:.3f}')
plt.plot(fpr_iss_female, tpr_iss_female, color='darkorange', lw=2, label=f'ROC AUC ISS = {roc_auc_iss_female:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve-Female')
plt.legend(loc='lower right')


In [ ]:
##allow for comparison between trauma severity scoring systems within a particular cohort, define helper fx to allow for calibration comparison
def bootstrap_calibration_curve(y_true, y_prob, n_bins=10, n_boot=1000, random_state=None):
    """
    1) Compute the original bin-based calibration curve.
    2) Bootstrap the dataset n_boot times, each time recalculating the bin-based
       fraction of positives (prob_true) and storing it.
    3) Return the original curve + 95% CI per bin (based on 2.5 and 97.5 percentiles).
    """
    # -----------------------------
    # Original calibration curve
    # -----------------------------
    # prob_true_orig, prob_pred_orig = calibration_curve(...) does binning internally.
    # But we want to fix n_bins and ensure consistent binning across bootstraps.
    # We'll do a manual binning approach here to keep consistent bin boundaries.

    # Define bin edges (equally spaced from 0 to 1)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    # Digitize predicted probabilities
    bin_indices = np.digitize(y_prob, bin_edges) - 1
    bin_indices[bin_indices == n_bins] = n_bins - 1  # cap any == n_bins to last bin

    # Prepare arrays to hold the original bin stats
    prob_pred_orig = np.zeros(n_bins)
    prob_true_orig = np.zeros(n_bins)
    counts_in_bin = np.zeros(n_bins, dtype=int)

    # Fill in the stats for each bin
    for i in range(n_bins):
        mask = (bin_indices == i)
        counts_in_bin[i] = np.sum(mask)
        if counts_in_bin[i] > 0:
            prob_pred_orig[i] = np.mean(y_prob[mask])   # mean predicted prob in this bin
            prob_true_orig[i] = np.mean(y_true[mask])   # fraction of positives (actual)
        else:
            # If bin is empty, set to NaN
            prob_pred_orig[i] = np.nan
            prob_true_orig[i] = np.nan

    # Remove empty bins (NaN) from the original arrays
    valid_mask = ~np.isnan(prob_pred_orig)
    prob_pred_orig = prob_pred_orig[valid_mask]
    prob_true_orig = prob_true_orig[valid_mask]

    # -----------------------------
    # Bootstrap to get CIs
    # -----------------------------
    rng = np.random.RandomState(random_state) if random_state else np.random

    # We'll store the fraction of positives (prob_true) for each bin in each bootstrap
    # but only for the bins that were valid in the original data
    boot_prob_true = np.zeros((n_boot, sum(valid_mask)))

    n_data = len(y_true)
    data_idx = np.arange(n_data)

    for b in range(n_boot):
        # Sample with replacement
        sample_indices = rng.randint(0, n_data, size=n_data)
        y_true_b = y_true[sample_indices]
        y_prob_b = y_prob[sample_indices]

        # Repeat the binning steps
        bin_indices_b = np.digitize(y_prob_b, bin_edges) - 1
        bin_indices_b[bin_indices_b == n_bins] = n_bins - 1

        prob_true_b = np.zeros(n_bins)
        for i in range(n_bins):
            mask_b = (bin_indices_b == i)
            if np.sum(mask_b) > 0:
                prob_true_b[i] = np.mean(y_true_b[mask_b])
            else:
                prob_true_b[i] = np.nan

        # filter to only valid bins
        prob_true_b = prob_true_b[valid_mask]
        boot_prob_true[b, :] = prob_true_b

    # Compute 2.5th and 97.5th percentile per bin (column-wise)
    lower_ci = np.nanpercentile(boot_prob_true, 2.5, axis=0)
    upper_ci = np.nanpercentile(boot_prob_true, 97.5, axis=0)

    return prob_pred_orig, prob_true_orig, lower_ci, upper_ci

# Extract predictions and true labels
y_true = np.array(female_df['y_true'])

y_prob_gbo = np.array(female_df['y_prob_model'])
y_prob_triss = np.array(female_df['y_prob_triss'])
y_prob_iss = np.array(female_df['y_prob_iss'])

# Helper function to get sorted calibration data
def get_bootstrap_calibration_data(y_true, y_prob, label, color, n_bins=10, n_boot=1000, random_state=42):
    prob_pred, prob_true, lower_ci, upper_ci = bootstrap_calibration_curve(
        y_true, y_prob, n_bins=n_bins, n_boot=n_boot, random_state=random_state
    )
    sort_idx = np.argsort(prob_pred)
    return {
        "x": prob_pred[sort_idx],
        "y": prob_true[sort_idx],
        "lower": lower_ci[sort_idx],
        "upper": upper_ci[sort_idx],
        "label": label,
        "color": color
    }

# Get calibration data for each method
calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='green')
calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

# Compute Brier Scores
brier_gbo = brier_score_loss(y_true, y_prob_gbo)
brier_triss = brier_score_loss(y_true, y_prob_triss)
brier_iss = brier_score_loss(y_true, y_prob_iss)

# Print Brier scores
print(f"Brier Score - ML Model: {brier_gbo:.4f}")
print(f"Brier Score - TRISS:     {brier_triss:.4f}")
print(f"Brier Score - ISS:       {brier_iss:.4f}")

# Plot the reliability diagram
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

for calib in [calib_gbo
              #, calib_triss
              #, calib_iss
              ]:
    plt.plot(calib["x"], calib["y"], marker='o', label=f'{calib["label"]}', color=calib["color"])
    plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Mortality Rate')
plt.title('Reliability Diagram with 95% CI - Female')
plt.legend(loc='best')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.grid(True)

# Optional: save and download
plt.show()


In [ ]:
##now compare DCA between scoring systems
# 6. Decision Curve
# ========================================

def net_benefit(y_true, y_prob, thresholds):
    """
    NetBenefit = (TP/N) - (FP/N)*(threshold/(1-threshold))
    """
    N = len(y_true)
    NB = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        TP = np.sum((y_true == 1) & (y_pred == 1))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        if t == 1.0:
            nb_t = 0
        else:
            nb_t = (TP / N) - (FP / N) * (t / (1 - t))
        NB.append(nb_t)
    return NB

# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)



# Make sure y_true and predicted probs are arrays
y_true_array_dc = np.array(female_df['y_true']).flatten()
y_prob_gbo_array_dc = np.array(female_df['y_prob_model'])
iss_probs_dc = np.array(female_df['y_prob_iss'])
X_TRISS_dc = female_df['y_prob_triss']

# Net benefit for your new model
NB_model = net_benefit(y_true_array_dc, y_prob_gbo_array_dc, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array_dc, iss_probs_dc, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array_dc, X_TRISS_dc.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve -Female')
plt.legend(loc='best')
plt.ylim([-0.04, 0.04])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
##compare MLISS to ISS and TRISS for females

pairs = {
    "ISS": female_df['y_prob_iss'],
    "TRISS": female_df['y_prob_triss']
}

base_seed = 42

for i, (var_name, var_array) in enumerate(pairs.items()):
    results = paired_bootstrap_auc_test(
        y_true=female_df['y_true'],
        predA=female_df['y_prob_model'],
        predB=var_array,
        n_boot=2000,
        alpha=0.025,
        random_state=base_seed + i
    )

    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} --- in female patients")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")


In [ ]:
##same thing as above for males only

pairs = {
    "ISS": male_df['y_prob_iss'],
    "TRISS": male_df['y_prob_triss']
}

for var_name, var_array in pairs.items():
    results = paired_bootstrap_auc_test(
        y_true=male_df['y_true'],
        predA=male_df['y_prob_model'],
        predB=var_array,
        n_boot=2000,      # or more for higher precision
        alpha=(0.05),
        random_state=42
    )
    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} ---")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")


In [ ]:
##same thing as above for Black patients

# Step 1: Map MORTALITY to 0/1 and subset complications_test_df to match X_test_tensor
complications_test_df = complications_df.loc[X_test.index].copy()
complications_test_df['MORTALITY'] = complications_test_df['MORTALITY'].map({'No': 0, 'Yes': 1})

# Step 2: Ensure data types are numeric and clean
complications_test_df = complications_test_df.dropna(subset=['MORTALITY'])
y_true_all = complications_test_df['MORTALITY'].astype(int).values

# Step 3: Scale the test data if not already done
# X_test_scaled = scaler.transform(X_test_tensor)  # Make sure you use the same scaler from training

# Step 4: Predict for entire test set once
y_prob_model = y_prob_gbo_mtp

# Step 5: Attach probabilities and true labels to complications_test_df for slicing
complications_test_df['y_true'] = y_true_all
complications_test_df['y_prob_model'] = y_prob_model
complications_test_df['y_prob_iss'] = iss_probs.flatten()
complications_test_df['y_prob_triss'] = X_TRISS.values.flatten()

# Step 6: Define evaluation function
def evaluate_group(df, race_group_name, is_black=True):
    if is_black:
        group_df = df[df['RACE'] == race_group_name]
        group_label = race_group_name
    else:
        group_df = df[df['RACE'] != race_group_name]
        group_label = f"Non-{race_group_name}"

    y_true = group_df['y_true']

    return {
        'Group': group_label,
        'AUROC_ML': roc_auc_score(y_true, group_df['y_prob_model']),
        'AUROC_ISS': roc_auc_score(y_true, group_df['y_prob_iss']),
        'AUROC_TRISS': roc_auc_score(y_true, group_df['y_prob_triss']),
        'Brier_ML': brier_score_loss(y_true, group_df['y_prob_model']),
        'Brier_ISS': brier_score_loss(y_true, group_df['y_prob_iss']),
        'Brier_TRISS': brier_score_loss(y_true, group_df['y_prob_triss']),
        'N': len(group_df),
        'Positives': int((y_true == 1).sum()),
        'Negatives': int((y_true == 0).sum())
    }

# Step 7: Run evaluation
results_black = evaluate_group(complications_test_df, 'BLACK', is_black=True)
results_nonblack = evaluate_group(complications_test_df, 'BLACK', is_black=False)


# Step 8: Print results
for result in [results_black, results_nonblack]:
    print(f"=== {result['Group']} ===")
    print(f"AUROC (ML):    {result['AUROC_ML']:.3f}")
    print(f"AUROC (ISS):   {result['AUROC_ISS']:.3f}")
    print(f"AUROC (TRISS): {result['AUROC_TRISS']:.3f}")
    print(f"Brier (ML):    {result['Brier_ML']:.3f}")
    print(f"Brier (ISS):   {result['Brier_ISS']:.3f}")
    print(f"Brier (TRISS): {result['Brier_TRISS']:.3f}")
    print(f"N:             {result['N']}")
    print(f"Positives:     {result['Positives']}")
    print(f"Negatives:     {result['Negatives']}\n")

      # Calculate AUROC delta between Male and Female for each method
delta_auroc_model = results_black['AUROC_ML'] - results_nonblack['AUROC_ML']
delta_auroc_iss = results_black['AUROC_ISS'] - results_nonblack['AUROC_ISS']
delta_auroc_triss = results_black['AUROC_TRISS'] - results_nonblack['AUROC_TRISS']

# Print deltas
print("=== AUROC Deltas (Black - Nonblack) ===")
print(f"Model: {delta_auroc_model:.4f}")
print(f"ISS:   {delta_auroc_iss:.4f}")
print(f"TRISS: {delta_auroc_triss:.4f}")

In [ ]:
##same thing as above but now for black patients
black_df= complications_test_df[complications_test_df['RACE'] == 'BLACK']
nonblack_df= complications_test_df[complications_test_df['RACE'] != 'BLACK']
black_df.shape

In [ ]:
##same thing as above but now for black patients
# Assume your DataFrame is named df
# Replace 'ISS_05' and 'MORTALITY' with the appropriate column names if different
predicted_prob_iss_black=black_df['y_prob_iss']
predicted_prob_triss_black=black_df['y_prob_triss']
predicted_prob_gbo_black=black_df['y_prob_model']
true_label_black=black_df['y_true']

# Calculate the FPR, TPR, and thresholds
fpr_iss_black, tpr_iss_black, thresholds_iss_black = roc_curve(true_label_black, predicted_prob_iss_black)

##now TRISS
fpr_triss_black, tpr_triss_black, thresholds_triss_black = roc_curve(true_label_black, predicted_prob_triss_black)

##and MLISS
fpr_gbo_black, tpr_gbo_black, thresholds_gbo_black = roc_curve(true_label_black, predicted_prob_gbo_black)

# Calculate the area under the ROC curve (AUROC)
roc_auc_iss_black = auc(fpr_iss_black, tpr_iss_black)

##now TRISS
roc_auc_triss_black = auc(fpr_triss_black, tpr_triss_black)

##and MLISS
roc_auc_gbo_black = auc(fpr_gbo_black, tpr_gbo_black)


# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo_black, tpr_gbo_black, color='b', lw=2, label=f'ROC curve (area = {roc_auc_gbo_black:.3f})')
plt.plot(fpr_triss_black, tpr_triss_black, color='green', lw=2, label=f'ROC AUC TRISS = {roc_auc_triss_black:.3f}')
plt.plot(fpr_iss_black, tpr_iss_black, color='darkorange', lw=2, label=f'ROC AUC ISS = {roc_auc_iss_black:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve-Black')
plt.legend(loc='lower right')

In [ ]:
##same thing as above but now for black patients

# Extract predictions and true labels
y_true = np.array(black_df['y_true'])
y_prob_gbo = np.array(black_df['y_prob_model'])
y_prob_triss = np.array(black_df['y_prob_triss'])
y_prob_iss = np.array(black_df['y_prob_iss'])

# Get calibration data for each method
calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='green')
calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

# Compute Brier Scores
brier_gbo = brier_score_loss(y_true, y_prob_gbo)
brier_triss = brier_score_loss(y_true, y_prob_triss)
brier_iss = brier_score_loss(y_true, y_prob_iss)

# Print Brier scores
print(f"Brier Score - ML Model: {brier_gbo:.4f}")
print(f"Brier Score - TRISS:     {brier_triss:.4f}")
print(f"Brier Score - ISS:       {brier_iss:.4f}")

# Plot the reliability diagram
plt.figure(figsize=(8,6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

for calib in [calib_gbo
               #, calib_triss, calib_iss
              ]:
    plt.plot(calib["x"], calib["y"], marker='o', label=calib["label"], color=calib["color"])
    plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Mortality Rate')
plt.title('Reliability Diagram with 95% CI - Black Patients')
plt.legend(loc='best')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for black patients
# 6. Decision Curve (Amended with ISS and TRISS)
# ========================================


# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)



# Make sure y_true and predicted probs are arrays
y_true_array_dc = np.array(black_df['y_true']).flatten()
y_prob_gbo_array_dc = np.array(black_df['y_prob_model'])
iss_probs_dc = np.array(black_df['y_prob_iss'])
X_TRISS_dc = black_df['y_prob_triss']

# Net benefit for your new model
NB_model = net_benefit(y_true_array_dc, y_prob_gbo_array_dc, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array_dc, iss_probs_dc, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array_dc, X_TRISS_dc.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis-Black')
plt.legend(loc='best')
plt.ylim([-0.1, 0.1])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for black patients

# Suppose you have these arrays
# Y_test: ground truth labels (0 or 1)
# ML_preds: predictions from your ML model
# ISS_vals, pTRISS_vals, TRISS_vals: numeric scores

pairs = {
    "ISS": black_df['y_prob_iss'],
    "TRISS": black_df['y_prob_triss']
}

base_seed = 42

for i, (var_name, var_array) in enumerate(pairs.items()):
    results = paired_bootstrap_auc_test(
        y_true=black_df['y_true'],
        predA=black_df['y_prob_model'],
        predB=var_array,
        n_boot=2000,      # or more for higher precision
        alpha=0.025,      # 95% CI in your function
        random_state=base_seed + i
    )
    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} --- in Black patients")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")

In [ ]:
##same thing as above but now for Hispanic patients

# Step 1: Prepare complications_test_df with mortality mapped to 0/1
complications_test_df = complications_df.loc[X_test.index].copy()
complications_test_df['MORTALITY'] = complications_test_df['MORTALITY'].map({'No': 0, 'Yes': 1})
complications_test_df = complications_test_df.dropna(subset=['MORTALITY'])
y_true_all = complications_test_df['MORTALITY'].astype(int).values

# Step 2: Generate model predictions (already trained + scaled)
# X_test_scaled = scaler.transform(X_test_tensor)
y_prob_model = y_prob_gbo_mtp

# Step 3: Add predictions to dataframe
complications_test_df['y_true'] = y_true_all
complications_test_df['y_prob_model'] = y_prob_model
complications_test_df['y_prob_iss'] = iss_probs.flatten()
complications_test_df['y_prob_triss'] = X_TRISS.values.flatten()

# Step 4: Evaluation function for any group
def evaluate_all_models(df, group_name):
    group_df = df[df['ETHNICITY'] == group_name]
    y_true = group_df['y_true']

    return {
        'Group': group_name,
        'AUROC_ML': roc_auc_score(y_true, group_df['y_prob_model']),
        'AUROC_ISS': roc_auc_score(y_true, group_df['y_prob_iss']),
        'AUROC_TRISS': roc_auc_score(y_true, group_df['y_prob_triss']),
        'Brier_ML': brier_score_loss(y_true, group_df['y_prob_model']),
        'Brier_ISS': brier_score_loss(y_true, group_df['y_prob_iss']),
        'Brier_TRISS': brier_score_loss(y_true, group_df['y_prob_triss']),
        'N': len(group_df),
        'Positives': int((y_true == 1).sum()),
        'Negatives': int((y_true == 0).sum())
    }

# Step 5: Run evaluation for Hispanic v No
results_hisp = evaluate_all_models(complications_test_df, 'H')
results_nothisp = evaluate_all_models(complications_test_df, 'N')

# Step 6: Display results
for result in [results_hisp, results_nothisp]:
    print(f"=== {result['Group']} ===")
    print(f"AUROC (ML):    {result['AUROC_ML']:.3f}")
    print(f"AUROC (ISS):   {result['AUROC_ISS']:.3f}")
    print(f"AUROC (TRISS): {result['AUROC_TRISS']:.3f}")
    print(f"Brier (ML):    {result['Brier_ML']:.3f}")
    print(f"Brier (ISS):   {result['Brier_ISS']:.3f}")
    print(f"Brier (TRISS): {result['Brier_TRISS']:.3f}")
    print(f"N:             {result['N']}")
    print(f"Positives:     {result['Positives']}")
    print(f"Negatives:     {result['Negatives']}\n")

    # Calculate AUROC delta between Male and Female for each method
delta_auroc_model = results_hisp['AUROC_ML'] - results_nothisp['AUROC_ML']
delta_auroc_iss = results_hisp['AUROC_ISS'] - results_nothisp['AUROC_ISS']
delta_auroc_triss = results_hisp['AUROC_TRISS'] - results_nothisp['AUROC_TRISS']

# Print deltas
print("=== AUROC Deltas (Hisp - Not hisp) ===")
print(f"Model: {delta_auroc_model:.4f}")
print(f"ISS:   {delta_auroc_iss:.4f}")
print(f"TRISS: {delta_auroc_triss:.4f}")


In [ ]:
##same thing as above but now for Hispanic patients
hisp_df= complications_test_df[complications_test_df['ETHNICITY'] == 'H']
nonhisp_df= complications_test_df[complications_test_df['ETHNICITY'] != 'H']
hisp_df.shape

In [ ]:
##same thing as above but now for Hispanic patients

predicted_prob_iss_hispanic=hisp_df['y_prob_iss']
predicted_prob_triss_hispanic=hisp_df['y_prob_triss']
predicted_prob_gbo_hispanic=hisp_df['y_prob_model']
true_label_hispanic=hisp_df['y_true']

# Calculate the FPR, TPR, and thresholds
fpr_iss_hispanic, tpr_iss_hispanic, thresholds_iss_hispanic = roc_curve(true_label_hispanic, predicted_prob_iss_hispanic)

##now TRISS
fpr_triss_hispanic, tpr_triss_hispanic, thresholds_triss_hispanic = roc_curve(true_label_hispanic, predicted_prob_triss_hispanic)

##and MLISS
fpr_gbo_hispanic, tpr_gbo_hispanic, thresholds_gbo_hispanic = roc_curve(true_label_hispanic, predicted_prob_gbo_hispanic)

# Calculate the area under the ROC curve (AUROC)
roc_auc_iss_hispanic = auc(fpr_iss_hispanic, tpr_iss_hispanic)

##now TRISS
roc_auc_triss_hispanic = auc(fpr_triss_hispanic, tpr_triss_hispanic)

##and MLISS
roc_auc_gbo_hispanic = auc(fpr_gbo_hispanic, tpr_gbo_hispanic)


# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo_hispanic, tpr_gbo_hispanic, color='b', lw=2, label=f'ROC curve (area = {roc_auc_gbo_hispanic:.3f})')
plt.plot(fpr_triss_hispanic, tpr_triss_hispanic, color='green', lw=2, label=f'ROC AUC TRISS = {roc_auc_triss_hispanic:.3f}')
plt.plot(fpr_iss_hispanic, tpr_iss_hispanic, color='darkorange', lw=2, label=f'ROC AUC ISS = {roc_auc_iss_hispanic:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve-Hispanic')
plt.legend(loc='lower right')


In [ ]:
##same thing as above but now for Hispanic patients

# Extract predictions and true labels
y_true = np.array(hisp_df['y_true'])
y_prob_gbo = np.array(hisp_df['y_prob_model'])
y_prob_triss = np.array(hisp_df['y_prob_triss'])
y_prob_iss = np.array(hisp_df['y_prob_iss'])

# Get calibration data for each method
calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='green')
calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

# Compute Brier Scores
brier_gbo = brier_score_loss(y_true, y_prob_gbo)
brier_triss = brier_score_loss(y_true, y_prob_triss)
brier_iss = brier_score_loss(y_true, y_prob_iss)

# Print Brier scores
print(f"Brier Score - ML Model: {brier_gbo:.4f}")
print(f"Brier Score - TRISS:     {brier_triss:.4f}")
print(f"Brier Score - ISS:       {brier_iss:.4f}")

# Plot the reliability diagram
plt.figure(figsize=(8,6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

for calib in [calib_gbo
              # , calib_triss
              #, calib_iss
              ]:
    plt.plot(calib["x"], calib["y"], marker='o', label=calib["label"], color=calib["color"])
    plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Mortality Rate')
plt.title('Reliability Diagram with 95% CI - Hispanic Patients')
plt.legend(loc='best')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for Hispanic patients
# 6. Decision Curve (Amended with ISS and TRISS)
# ========================================


# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)



# Make sure y_true and predicted probs are arrays
y_true_array_dc = np.array(hisp_df['y_true']).flatten()
y_prob_gbo_array_dc = np.array(hisp_df['y_prob_model'])
iss_probs_dc = np.array(hisp_df['y_prob_iss'])
X_TRISS_dc = hisp_df['y_prob_triss']

# Net benefit for your new model
NB_model = net_benefit(y_true_array_dc, y_prob_gbo_array_dc, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array_dc, iss_probs_dc, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array_dc, X_TRISS_dc.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis-Hispanic')
plt.legend(loc='best')
plt.ylim([-0.05, 0.05])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for Hispanic patients
pairs = {
    "ISS": hisp_df['y_prob_iss'],
    "TRISS": hisp_df['y_prob_triss']
}

base_seed = 42

for i, (var_name, var_array) in enumerate(pairs.items()):
    results = paired_bootstrap_auc_test(
        y_true=hisp_df['y_true'],
        predA=hisp_df['y_prob_model'],
        predB=var_array,
        n_boot=2000,
        alpha=0.025,              # 95% CI
        random_state=base_seed + i
    )
    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} --- in Hispanic patients")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")

In [ ]:
##same thing as above but now for self-pay patients

# Step 1: Map MORTALITY to 0/1 and subset complications_test_df to match X_test_tensor
complications_test_df = complications_df.loc[X_test.index].copy()
complications_test_df['MORTALITY'] = complications_test_df['MORTALITY'].map({'No': 0, 'Yes': 1})

# Step 2: Ensure data types are numeric and clean
complications_test_df = complications_test_df.dropna(subset=['MORTALITY'])
y_true_all = complications_test_df['MORTALITY'].astype(int).values

# Step 3: Scale the test data if not already done
# X_test_scaled = scaler.transform(X_test_tensor)  # Make sure you use the same scaler from training

# Step 4: Predict for entire test set once
y_prob_model = y_prob_gbo_mtp

# Step 5: Attach probabilities and true labels to complications_test_df for slicing
complications_test_df['y_true'] = y_true_all
complications_test_df['y_prob_model'] = y_prob_model
complications_test_df['y_prob_iss'] = iss_probs.flatten()
complications_test_df['y_prob_triss'] = X_TRISS.values.flatten()

# Step 6: Define evaluation function
def evaluate_group(df, pay_group_name, is_uninsured=True):
    if is_uninsured:
        group_df = df[df['PRIMARYMETHODPAYMENT'] == pay_group_name]
        group_label = pay_group_name
    else:
        group_df = df[df['PRIMARYMETHODPAYMENT'] != pay_group_name]
        group_label = f"Non-{pay_group_name}"

    y_true = group_df['y_true']

    return {
        'Group': group_label,
        'AUROC_ML': roc_auc_score(y_true, group_df['y_prob_model']),
        'AUROC_ISS': roc_auc_score(y_true, group_df['y_prob_iss']),
        'AUROC_TRISS': roc_auc_score(y_true, group_df['y_prob_triss']),
        'Brier_ML': brier_score_loss(y_true, group_df['y_prob_model']),
        'Brier_ISS': brier_score_loss(y_true, group_df['y_prob_iss']),
        'Brier_TRISS': brier_score_loss(y_true, group_df['y_prob_triss']),
        'N': len(group_df),
        'Positives': int((y_true == 1).sum()),
        'Negatives': int((y_true == 0).sum())
    }

# Step 7: Run evaluation
results_uninsured = evaluate_group(complications_test_df, 'SELF-PAY', is_uninsured=True)
results_insured = evaluate_group(complications_test_df, 'SELF-PAY', is_uninsured=False)


# Step 8: Print results
for result in [results_uninsured, results_insured]:
    print(f"=== {result['Group']} ===")
    print(f"AUROC (ML):    {result['AUROC_ML']:.3f}")
    print(f"AUROC (ISS):   {result['AUROC_ISS']:.3f}")
    print(f"AUROC (TRISS): {result['AUROC_TRISS']:.3f}")
    print(f"Brier (ML):    {result['Brier_ML']:.3f}")
    print(f"Brier (ISS):   {result['Brier_ISS']:.3f}")
    print(f"Brier (TRISS): {result['Brier_TRISS']:.3f}")
    print(f"N:             {result['N']}")
    print(f"Positives:     {result['Positives']}")
    print(f"Negatives:     {result['Negatives']}\n")

      # Calculate AUROC delta between insured and noninsured for each method
delta_auroc_model = results_uninsured['AUROC_ML'] - results_insured['AUROC_ML']
delta_auroc_iss = results_uninsured['AUROC_ISS'] - results_insured['AUROC_ISS']
delta_auroc_triss = results_uninsured['AUROC_TRISS'] - results_insured['AUROC_TRISS']

# Print deltas
print("=== AUROC Deltas (Insured - Uninsured) ===")
print(f"Model: {delta_auroc_model:.4f}")
print(f"ISS:   {delta_auroc_iss:.4f}")
print(f"TRISS: {delta_auroc_triss:.4f}")

In [ ]:
##same thing as above but now for self-pay patients
selfpay_df= complications_test_df[complications_test_df['PRIMARYMETHODPAYMENT'] == 'SELF-PAY']
insured_df= complications_test_df[complications_test_df['PRIMARYMETHODPAYMENT'] != 'SELF-PAY']
selfpay_df.shape

In [ ]:
##same thing as above but now for self-pay patients
predicted_prob_iss_selfpay=selfpay_df['y_prob_iss']
predicted_prob_triss_selfpay=selfpay_df['y_prob_triss']
predicted_prob_gbo_selfpay=selfpay_df['y_prob_model']
true_label_selfpay=selfpay_df['y_true']
# Calculate the FPR, TPR, and thresholds
fpr_iss_selfpay, tpr_iss_selfpay, thresholds_iss_selfpay = roc_curve(true_label_selfpay, predicted_prob_iss_selfpay)

##now TRISS
fpr_triss_selfpay, tpr_triss_selfpay, thresholds_triss_selfpay = roc_curve(true_label_selfpay, predicted_prob_triss_selfpay)

##and MLISS
fpr_gbo_selfpay, tpr_gbo_selfpay, thresholds_gbo_selfpay = roc_curve(true_label_selfpay, predicted_prob_gbo_selfpay)

# Calculate the area under the ROC curve (AUROC)
roc_auc_iss_selfpay = auc(fpr_iss_selfpay, tpr_iss_selfpay)

##now TRISS
roc_auc_triss_selfpay = auc(fpr_triss_selfpay, tpr_triss_selfpay)

##and MLISS
roc_auc_gbo_selfpay = auc(fpr_gbo_selfpay, tpr_gbo_selfpay)



# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo_selfpay, tpr_gbo_selfpay, color='b', lw=2, label=f'ROC curve (area = {roc_auc_gbo_selfpay:.3f})')
plt.plot(fpr_triss_selfpay, tpr_triss_selfpay, color='green', lw=2, label=f'ROC AUC TRISS = {roc_auc_triss_selfpay:.3f}')
plt.plot(fpr_iss_selfpay, tpr_iss_selfpay, color='darkorange', lw=2, label=f'ROC AUC ISS = {roc_auc_iss_selfpay:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve-Selfpay')
plt.legend(loc='lower right')

In [ ]:
##same thing as above but now for self-pay patients

# Extract predictions and true labels
y_true = np.array(selfpay_df['y_true'])
y_prob_gbo = np.array(selfpay_df['y_prob_model'])
y_prob_triss = np.array(selfpay_df['y_prob_triss'])
y_prob_iss = np.array(selfpay_df['y_prob_iss'])

# Get calibration data for each method
calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='green')
calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

# Compute Brier Scores
brier_gbo = brier_score_loss(y_true, y_prob_gbo)
brier_triss = brier_score_loss(y_true, y_prob_triss)
brier_iss = brier_score_loss(y_true, y_prob_iss)

# Print Brier scores
print(f"Brier Score - ML Model: {brier_gbo:.4f}")
print(f"Brier Score - TRISS:     {brier_triss:.4f}")
print(f"Brier Score - ISS:       {brier_iss:.4f}")

# Plot the reliability diagram
plt.figure(figsize=(8,6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

for calib in [calib_gbo
               #, calib_triss, calib_iss
              ]:
    plt.plot(calib["x"], calib["y"], marker='o', label=calib["label"], color=calib["color"])
    plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Mortality Rate')
plt.title('Reliability Diagram with 95% CI - Self-pay Patients')
plt.legend(loc='best')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for self-pay patients
# 6. Decision Curve (Amended with ISS and TRISS)
# ========================================


# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)



# Make sure y_true and predicted probs are arrays
y_true_array_dc = np.array(selfpay_df['y_true']).flatten()
y_prob_gbo_array_dc = np.array(selfpay_df['y_prob_model'])
iss_probs_dc = np.array(selfpay_df['y_prob_iss'])
X_TRISS_dc = selfpay_df['y_prob_triss']

# Net benefit for your new model
NB_model = net_benefit(y_true_array_dc, y_prob_gbo_array_dc, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array_dc, iss_probs_dc, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array_dc, X_TRISS_dc.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis-Selfpay')
plt.legend(loc='best')
plt.ylim([-0.2, 0.2])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now for self-pay patients

pairs = {
    "ISS": selfpay_df['y_prob_iss'],
    "TRISS": selfpay_df['y_prob_triss']
}

base_seed = 42

for i, (var_name, var_array) in enumerate(pairs.items()):
    results = paired_bootstrap_auc_test(
        y_true=selfpay_df['y_true'],
        predA=selfpay_df['y_prob_model'],
        predB=var_array,
        n_boot=2000,      # or more for higher precision
        alpha=0.025,      # 95% CI in your function
        random_state=base_seed + i
    )
    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} --- in uninsured patients")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")

In [ ]:
##same thing as above but now in ISS>30 patients

# Step 1: Map MORTALITY to 0/1 and subset complications_test_df to match X_test_tensor
complications_test_df = complications_df.loc[X_test.index].copy()
complications_test_df['MORTALITY'] = complications_test_df['MORTALITY'].map({'No': 0, 'Yes': 1})

# Step 2: Ensure data types are numeric and clean
complications_test_df = complications_test_df.dropna(subset=['MORTALITY', 'ISS_05']).copy()
y_true_all = complications_test_df['MORTALITY'].astype(int).values

# Step 3: Predict for entire test set once
y_prob_model = y_prob_gbo_mtp

# Step 4: Attach predictions and labels
complications_test_df['y_true'] = y_true_all
complications_test_df['y_prob_model'] = y_prob_model
complications_test_df['y_prob_iss'] = iss_probs.flatten()
complications_test_df['y_prob_triss'] = X_TRISS.values.flatten()

# Step 5: Define evaluation function for ISS group
def evaluate_iss_group(df, min_iss=None, max_iss=None, exact_iss=None):
    if exact_iss is not None:
        group_df = df[df['ISS_05'] == exact_iss]
        label = f"ISS {exact_iss}"
    elif min_iss is not None and max_iss is not None:
        group_df = df[(df['ISS_05'] >= min_iss) & (df['ISS_05'] <= max_iss)]
        label = f"ISS {min_iss}-{max_iss}"
    else:
        raise ValueError("Provide either min_iss and max_iss, or exact_iss.")

    y_true = group_df['y_true']
    return {
        'Group': label,
        'AUROC_ML': roc_auc_score(y_true, group_df['y_prob_model']),
        'AUROC_ISS': roc_auc_score(y_true, group_df['y_prob_iss']),
        'AUROC_TRISS': roc_auc_score(y_true, group_df['y_prob_triss']),
        'Brier_ML': brier_score_loss(y_true, group_df['y_prob_model']),
        'Brier_ISS': brier_score_loss(y_true, group_df['y_prob_iss']),
        'Brier_TRISS': brier_score_loss(y_true, group_df['y_prob_triss']),
        'N': len(group_df),
        'Positives': int((y_true == 1).sum()),
        'Negatives': int((y_true == 0).sum())
    }

# Step 6: Evaluate each ISS group
results_iss_0_29 = evaluate_iss_group(complications_test_df, 0, 29)
results_iss_30_74 = evaluate_iss_group(complications_test_df, 30, 74)
results_iss_75 = evaluate_iss_group(complications_test_df, exact_iss=75)

# Step 7: Print results
for result in [results_iss_0_29, results_iss_30_74, results_iss_75]:
    print(f"=== {result['Group']} ===")
    print(f"AUROC (ML):    {result['AUROC_ML']:.3f}")
    print(f"AUROC (ISS):   {result['AUROC_ISS']:.3f}")
    print(f"AUROC (TRISS): {result['AUROC_TRISS']:.3f}")
    print(f"Brier (ML):    {result['Brier_ML']:.3f}")
    print(f"Brier (ISS):   {result['Brier_ISS']:.3f}")
    print(f"Brier (TRISS): {result['Brier_TRISS']:.3f}")
    print(f"N:             {result['N']}")
    print(f"Positives:     {result['Positives']}")
    print(f"Negatives:     {result['Negatives']}\n")

In [ ]:
##same thing as above but now in ISS>30 patients

iss0to29_df = complications_test_df[(complications_test_df['ISS_05'] >= 0) & (complications_test_df['ISS_05'] <= 29)]
iss30to74_df = complications_test_df[(complications_test_df['ISS_05'] >= 30) & (complications_test_df['ISS_05'] <= 74)]
iss75_df = complications_test_df[complications_test_df['ISS_05'] == 75]

print("ISS 0-29 shape:", iss0to29_df.shape)
print("ISS 30-74 shape:", iss30to74_df.shape)
print("ISS 75 shape:", iss75_df.shape)

In [ ]:
##same thing as above but now in ISS>30 patients

predicted_prob_iss_30to74 = iss30to74_df['y_prob_iss']
predicted_prob_triss_30to74 = iss30to74_df['y_prob_triss']
predicted_prob_gbo_30to74 = iss30to74_df['y_prob_model']
true_label_30to74 = iss30to74_df['y_true']

fpr_iss_30to74, tpr_iss_30to74, _ = roc_curve(true_label_30to74, predicted_prob_iss_30to74)
fpr_triss_30to74, tpr_triss_30to74, _ = roc_curve(true_label_30to74, predicted_prob_triss_30to74)
fpr_gbo_30to74, tpr_gbo_30to74, _ = roc_curve(true_label_30to74, predicted_prob_gbo_30to74)

roc_auc_iss_30to74 = auc(fpr_iss_30to74, tpr_iss_30to74)
roc_auc_triss_30to74 = auc(fpr_triss_30to74, tpr_triss_30to74)
roc_auc_gbo_30to74 = auc(fpr_gbo_30to74, tpr_gbo_30to74)

plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo_30to74, tpr_gbo_30to74, color='b', lw=2, label=f'ML Model (AUC = {roc_auc_gbo_30to74:.3f})')
plt.plot(fpr_triss_30to74, tpr_triss_30to74, color='green', lw=2, label=f'TRISS (AUC = {roc_auc_triss_30to74:.3f})')
plt.plot(fpr_iss_30to74, tpr_iss_30to74, color='darkorange', lw=2, label=f'ISS (AUC = {roc_auc_iss_30to74:.3f})')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - ISS 30-74')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now in ISS>30 patients

# Define ISS groups
iss_groups = {
    "ISS 30-74": iss30to74_df
}

for group_name, df in iss_groups.items():
    print(f"\n=== {group_name} ===")

    # Extract predictions and true labels
    y_true = np.array(df['y_true'])
    y_prob_gbo = np.array(df['y_prob_model'])
    y_prob_triss = np.array(df['y_prob_triss'])
    y_prob_iss = np.array(df['y_prob_iss'])

    # Skip groups with insufficient outcome variation
    if len(df) == 0 or len(np.unique(y_true)) < 2:
        print("Skipping: subgroup has insufficient data or only one outcome class.")
        continue

    # Get calibration data for each method
    calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
    calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='green')
    calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

    # Compute Brier Scores
    brier_gbo = brier_score_loss(y_true, y_prob_gbo)
    brier_triss = brier_score_loss(y_true, y_prob_triss)
    brier_iss = brier_score_loss(y_true, y_prob_iss)

    # Print Brier scores
    print(f"Brier Score - ML Model: {brier_gbo:.4f}")
    print(f"Brier Score - TRISS:     {brier_triss:.4f}")
    print(f"Brier Score - ISS:       {brier_iss:.4f}")

    # Plot the reliability diagram
    plt.figure(figsize=(8,6))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

    for calib in [calib_gbo
                   , calib_triss, calib_iss
                  ]:
        plt.plot(calib["x"], calib["y"], marker='o', label=calib["label"], color=calib["color"])
        plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Observed Mortality Rate')
    plt.title(f'Reliability Diagram with 95% CI - {group_name}')
    plt.legend(loc='best')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True)
    plt.show()

In [ ]:
##same thing as above but now in ISS>30 patients

# 6. Decision Curve (Amended with ISS and TRISS)
# ========================================


# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)



# Make sure y_true and predicted probs are arrays
y_true_array_dc = np.array(iss30to74_df['y_true']).flatten()
y_prob_gbo_array_dc = np.array(iss30to74_df['y_prob_model'])
iss_probs_dc = np.array(iss30to74_df['y_prob_iss'])
X_TRISS_dc = iss30to74_df['y_prob_triss']

# Net benefit for your new model
NB_model = net_benefit(y_true_array_dc, y_prob_gbo_array_dc, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array_dc, iss_probs_dc, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array_dc, X_TRISS_dc.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis-ISS 30-74')
plt.legend(loc='best')
plt.ylim([-0.5, 0.5])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
##same thing as above but now in ISS>30 patients

# === Define ISS bins ===
bins = [0, 30, 75, np.inf]
labels = ["<30", "30-74", "75"]
complications_test_df['ISS_BIN'] = pd.cut(complications_test_df['ISS_05'], bins=bins, labels=labels, right=False)
##this way the second bin will START with the second number, the second number will NOT be in the first bin
##currently the only thing that works is ISS>=30, excluding ISS=75

# === Loop over bins and run paired bootstrap ===
base_seed = 42

for i, group in enumerate(labels):
    df_sub = complications_test_df[complications_test_df['ISS_BIN'] == group]
    y_true = df_sub['y_true'].values
    ml_pred = df_sub['y_prob_model'].values
    iss_pred = df_sub['y_prob_iss'].values
    triss_pred = df_sub['y_prob_triss'].values

    print(f"\n=== ISS Stratum: {group} ===")
    print(f"N = {len(df_sub)}, Positives = {(y_true == 1).sum()}, Negatives = {(y_true == 0).sum()}")

    if len(df_sub) == 0 or len(np.unique(y_true)) < 2:
        print("Skipping: subgroup has insufficient data or only one outcome class.")
        continue

    # ML vs ISS
    res_iss = paired_bootstrap_auc_test(
        y_true, ml_pred, iss_pred,
        random_state=base_seed + i
    )
    print(f"--- ML vs ISS ---")
    print(f"AUC(ML): {res_iss['aucA']:.3f}, {res_iss['coverage']}% CI: [{res_iss['aucA_ci_lower']:.3f}, {res_iss['aucA_ci_upper']:.3f}]")
    print(f"AUC(ISS): {res_iss['aucB']:.3f}, {res_iss['coverage']}% CI: [{res_iss['aucB_ci_lower']:.3f}, {res_iss['aucB_ci_upper']:.3f}]")
    print(f"ΔAUC: {res_iss['baseline_diff']:.4f}, CI: [{res_iss['diff_ci_lower']:.4f}, {res_iss['diff_ci_upper']:.4f}], p = {res_iss['p_value']:.4f}")

    # ML vs TRISS
    res_triss = paired_bootstrap_auc_test(
        y_true, ml_pred, triss_pred,
        random_state=base_seed + i + 1000
    )
    print(f"--- ML vs TRISS ---")
    print(f"AUC(ML): {res_triss['aucA']:.3f}, {res_triss['coverage']}% CI: [{res_triss['aucA_ci_lower']:.3f}, {res_triss['aucA_ci_upper']:.3f}]")
    print(f"AUC(TRISS): {res_triss['aucB']:.3f}, {res_triss['coverage']}% CI: [{res_triss['aucB_ci_lower']:.3f}, {res_triss['aucB_ci_upper']:.3f}]")
    print(f"ΔAUC: {res_triss['baseline_diff']:.4f}, CI: [{res_triss['diff_ci_lower']:.4f}, {res_triss['diff_ci_upper']:.4f}], p = {res_triss['p_value']:.4f}")
